In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# %%bash 
# for f in drive/MyDrive/zipped_LOB_data/*; do
#     name=$(basename $f ".7z")
#     mkdir -p "data/extracted/$name"
#     7z x $f -o"data/extracted/$name" -y
# done


In [2]:
from pathlib import Path 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import torch 
import json
from datetime import datetime

DATA_ROOT = Path("data/extracted")
DATA_ROOT.exists()

True

In [3]:
symbol_dir = DATA_ROOT/"BAC_2026-04-01_2026-04-10_10"
orderbook_files = sorted(symbol_dir.glob("*orderbook_10.csv"))
message_files = sorted(symbol_dir.glob("*message_10.csv"))

In [ ]:
from deeplob_library.data import read_message_file,read_orderbook_file,combine_message_orderbook

message_df = read_message_file(message_files[0])
orderbook_df = read_orderbook_file(orderbook_files[0])
df = combine_message_orderbook(message_df,orderbook_df)

In [5]:
#get midprice 
from deeplob_library.data import get_smoothed_midprice_targets
df = get_smoothed_midprice_targets(df)











In [ ]:
from deeplob_library.data import get_lists_of_tensors

    
Xs,ys = get_lists_of_tensors(message_files,orderbook_files)

X_train = torch.cat(Xs[:-2],dim=0)
y_train = torch.cat(ys[:-2],dim=0)
X_val = Xs[-2]
y_val = ys[-2]
X_test = Xs[-1]
y_test = ys[-1]






In [8]:
from deeplob_library.data import normalize_train_val_test,make_target_labels

X_train,X_val,X_test = normalize_train_val_test(X_train,X_val,X_test)
y_train,y_val,y_test = make_target_labels(y_train,y_val,y_test)

In [ ]:
from torch.utils.data import TensorDataset,DataLoader
import torch.nn as nn
import torch.nn.functional as F

BATCH_SIZE = 1024

train_loader = DataLoader(TensorDataset(X_train,y_train),batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(TensorDataset(X_val,y_val),batch_size=BATCH_SIZE,shuffle=False)
test_loader = DataLoader(TensorDataset(X_test,y_test),batch_size=BATCH_SIZE,shuffle=False)

from deeplob_library.models import DeepLOB


In [10]:
RESULTS_DIR = Path("results")
def log_experiment(results:dict,
                    model:nn.Module = None,
                    config:dict = None,):
    t = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    #create results directory
    results_dir = RESULTS_DIR/f"{t}"
    results_dir.mkdir(parents=True,exist_ok=True)
    #save results as json
    with open(results_dir/"results.json","w") as f:
        json.dump(results,f)
    #save model
    if model is not None:
        torch.save(model.state_dict(),results_dir/"model.pth")
    #save config
    if config is not None:
        with open(results_dir/"config.json","w") as f:
            json.dump(config,f)


In [ ]:
from tqdm.auto import tqdm
import copy

CONFIG = {
    "LR":1e-3,
    "EPOCHS":10,
    "BATCH_SIZE":1024,
    "MODEL_NAME":"DeepLOB",
    "TICKER":"BAC",
}

#VARIABLES
LR = CONFIG["LR"]
EPOCHS = CONFIG["EPOCHS"]
BATCH_SIZE = CONFIG["BATCH_SIZE"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepLOB().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=LR)

from deeplob_library.train import train_model

model,history = train_model(model,
                            train_loader,
                            val_loader,
                            criterion,
                            optimizer,
                            device,
                            EPOCHS)




c:\Users\kyria\Desktop\cursor projects\DeepLOB_library\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Training:   0%|          | 0/10 [01:45<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
#test
from deeplob_library.train import test_model

test_loss,test_acc = test_model(model,test_loader,criterion,device)

